[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/cours/seance2_cours.ipynb)

# Séance 4.2 — Prédire une décision — qui va résilier ?

**Cours** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- transformer des colonnes de texte en variables utilisables par un modèle
- ajuster une régression logistique et lire une probabilité de départ
- lire une matrice de confusion et nommer les deux façons de se tromper
- calculer et interpréter justesse, précision, rappel et F1
- expliquer pourquoi la justesse est un piège sur des données déséquilibrées
- choisir un seuil de décision à partir d'un coût, pas d'une habitude

## Un opérateur, 7 043 abonnés, une seule question

Nouveau terrain : un opérateur télécom. Chaque ligne est un abonné, et la
colonne `churn` vaut **1 s'il a résilié**, 0 sinon.

> *« Qui va partir le mois prochain — et qui faut-il appeler ? »*

C'est une **prédiction binaire** : la réponse n'est plus un nombre, c'est une
décision.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

In [ ]:
tel = pd.read_csv(BASE + "churn.csv")   ## une ligne = un abonne

print(tel.shape)
tel.head(3)

### D'abord, nettoyer — rien n'a changé depuis le bloc 2

`total` est arrivé en texte : onze abonnés tout neufs n'ont pas encore de
facture cumulée.

In [ ]:
tel["total"] = pd.to_numeric(tel["total"], errors="coerce")   ## coerce -> NaN
print(tel["total"].isna().sum(), "valeurs vides ->", end=" ")   ## on VERIFIE

tel = tel.dropna(subset=["total"])   ## cette colonne seulement
print(len(tel), "abonnes conserves")

In [ ]:
# La moyenne d'une colonne de 0 et de 1, c'est la proportion de 1
print("taux de resiliation :", round(100 * tel["churn"].mean(), 1), "%")

(tel.groupby("contrat")["churn"].mean() * 100).round(1)   ## par type de contrat

**42,7 % chez les abonnés au mois, 2,8 % chez ceux engagés deux ans.**

Gardez ce tableau en tête : la séance 4.3 y reviendra pour en faire une
recommandation chiffrée.

## 1. Du texte vers des nombres

Un modèle multiplie des nombres : il ne sait pas quoi faire de `"mensuel"`.
`get_dummies` transforme chaque modalité en une colonne 0/1.

In [ ]:
y = tel["churn"]   ## la cible : 1 = parti
X = pd.get_dummies(tel.drop(columns=["churn"]), drop_first=True)

print(X.shape[1], "colonnes apres transformation")   ## 9 -> 14
list(X.columns)

> 💡 `drop_first=True` supprime une modalité par variable, celle qui devient
> la **référence** — exactement comme la modalité absente du tableau de
> régression de la séance 3.4. Si ce n'est ni `un_an` ni `deux_ans`, c'est
> forcément `mensuel` : la troisième colonne n'apporterait rien.

### Pourquoi cette étape n'était pas là en séance 4.1

Question légitime : en 4.1, on n'a jamais appelé `get_dummies`. Pourquoi
maintenant ?

Parce qu'en 4.1, **les deux variables d'entrée étaient déjà des nombres** :
`qte` et `nart` se comptent. Ici, `contrat`, `internet` et `paiement`
contiennent du texte.

La règle est la même dans les deux cas, et elle vaut pour **tout** modèle de
scikit-learn — régression linéaire comprise : un modèle multiplie chaque
variable par un coefficient, et on ne multiplie pas `"mensuel"` par 0,4. Ce
n'est pas la logistique qui exige `get_dummies`, c'est le fait d'avoir du
texte en entrée. Si la séance 4.1 avait voulu utiliser la colonne `pays`,
elle aurait dû faire exactement la même chose.

## 2. Ajuster une régression logistique

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)   ## stratify : voir plus bas

# stratify=y : garder le meme taux de resiliation des deux cotes
print(round(100 * y_tr.mean(), 1), "% de churn en apprentissage |",
      round(100 * y_te.mean(), 1), "% en test")

In [ ]:
# StandardScaler d'abord : l'anciennete va de 0 a 72, la facture cumulee
# de 18 a 8 700. Sans mise a l'echelle, l'optimisation converge mal.
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modele.fit(X_tr, y_tr)

# predict_proba renvoie deux colonnes : [proba de rester, proba de partir]
proba = modele.predict_proba(X_te)[:, 1]   ## l'indice 1 : la proba de partir
print("probabilite de depart des 5 premiers :", proba[:5].round(3))

La logistique ne répond pas « part » ou « reste » : elle donne une
**probabilité de départ**. C'est nous qui déciderons à partir de quel niveau
on agit — et ce choix est le sujet de la fin de séance.

## 3. Noter une prédiction binaire

En séance 4.1, l'erreur se mesurait en euros. Ici, il n'y a plus d'euros : une
prédiction est **juste ou fausse**. Mais il y a **deux façons** d'avoir faux,
et elles ne coûtent pas la même chose.

Croisons ce que le modèle a prédit avec ce qui s'est réellement passé.

In [ ]:
pred = modele.predict(X_te)   ## predict tranche a 0,50 par defaut

pd.DataFrame(confusion_matrix(y_te, pred),
             index=["reste vraiment", "part vraiment"],   ## la verite, en lignes
             columns=["predit reste", "predit part"])     ## le modele, en colonnes

### Les quatre cases, et leurs noms

|  | prédit : reste | prédit : part |
|---|---|---|
| **reste vraiment** | 1 377 — **vrai négatif** | 172 — **faux positif** |
| **part vraiment** | 255 — **faux négatif** | 306 — **vrai positif** |

- Un **faux positif** coûte un appel de rétention : 15 €.
- Un **faux négatif** coûte un client : il s'en va sans qu'on ait rien tenté.

Deux erreurs, deux additions très différentes. Aucune mesure unique ne peut
résumer ça — d'où les quatre qui suivent.

### Quatre mesures, toutes construites sur ces quatre cases

| Mesure | On divise | Elle répond à |
|---|---|---|
| **justesse** | (VN + VP) / total | quelle part de mes prédictions sont bonnes ? |
| **précision** | VP / (VP + FP) | parmi ceux que j'**appelle**, combien partaient vraiment ? |
| **rappel** | VP / (VP + FN) | parmi ceux qui **partent**, combien ai-je retrouvés ? |
| **F1** | 2 × précision × rappel / (précision + rappel) | les deux tiennent-elles ensemble ? |

La précision se divise par ceux qu'on **appelle**, le rappel par ceux qui
**partent**. C'est tout ce qui les sépare, et c'est ce qui change tout.

In [ ]:
print("justesse  :", round(accuracy_score(y_te, pred), 3))
print("precision :", round(precision_score(y_te, pred), 3))
print("rappel    :", round(recall_score(y_te, pred), 3))
print("F1        :", round(f1_score(y_te, pred), 3))

- **Justesse 0,798** : quatre prédictions sur cinq sont bonnes. Gardez ce
  chiffre, la section suivante le démolit.
- **Précision 0,640** : sur dix abonnés contactés, six allaient vraiment
  partir — quatre appels pour rien.
- **Rappel 0,545** : nous retrouvons un partant sur deux. L'autre moitié s'en
  va sans que personne ne l'appelle.
- **F1 0,589** : les deux précédentes résumées en un nombre.

### Pourquoi le F1 n'est pas une moyenne ordinaire

Précision et rappel se manipulent facilement, mais **en sens contraire** : pour
avoir un rappel de 1, il suffit d'appeler tout le monde. La précision tombe
alors à 0,27 — la proportion de partants dans le fichier.

Une moyenne ordinaire donnerait à cette stratégie absurde (1 + 0,27) / 2 =
**0,63**, mieux que notre modèle. Le F1, lui, la note **0,42** : il est tiré
vers le bas par la plus faible des deux.

> 💡 **Retenez ça du F1 :** il ne récompense pas un modèle qui sacrifie une
> mesure pour gonfler l'autre. Il faut être bon **des deux côtés**.

## 4. La justesse est un piège

79,8 % de bonnes réponses : de quoi présenter le projet en comité.

**Avant ça, une question :** quelle justesse obtiendrait un modèle qui prédit
que *personne* ne part ?

In [ ]:
# Le modele le plus bete du monde : il repond toujours "reste"
print("toujours predire 'reste' :", round(100 * (1 - y_te.mean()), 1), "%")   ## 73,4

**73,4 %.** Notre modèle ne gagne que **six points** sur un modèle qui ne fait
strictement rien — et qui, lui, ne sauverait aucun client.

C'est le piège des données **déséquilibrées** : quand une classe pèse trois
quarts du fichier, la justesse récompense le fait de toujours parier dessus.

Les trois autres mesures ne s'y laissent pas prendre. Un modèle qui ne prédit
jamais aucun départ n'a **aucun** vrai positif : sa précision, son rappel et
son F1 valent tous **zéro**. C'est ce qu'on vérifiera en fin de séance.

> ⚠️ **Ne jugez jamais une classification sur la seule justesse.** Demandez
> systématiquement trois choses : le score du modèle nul, la matrice de
> confusion, et le rappel.

## 5. Le seuil est une décision de gestion

`predict` a tranché à **0,50**, parce que c'est la valeur par défaut. Rien ne
l'impose. Regardons ce que d'autres seuils donnent.

In [ ]:
for seuil in [0.5, 0.4, 0.3, 0.2]:
    p = (proba > seuil).astype(int)   ## True/False -> 1/0
    print(f"seuil {seuil} : rappel {recall_score(y_te, p):.2f}  "
          f"precision {precision_score(y_te, p):.2f}  "
          f"F1 {f1_score(y_te, p):.2f}  contacts {p.sum()}")

Baisser le seuil retrouve plus de partants (rappel ↑) au prix de plus
d'appels inutiles (précision ↓). Le F1, qui tient les deux, bouge à peine :
de 0,59 à 0,62. **Aucun des réglages n'est « le bon » en soi** — cela dépend
de ce que coûte chaque erreur.

### Chiffrons

- un appel de rétention coûte **15 €**
- un client retenu rapporte **300 €** de marge sur l'année
- une relance convainc environ **30 %** des partants contactés

In [ ]:
for seuil in [0.5, 0.4, 0.3, 0.2, 0.1]:
    p = (proba > seuil).astype(int)
    vrais = ((p == 1) & (y_te == 1)).sum()   ## partants effectivement rattrapes
    gain = vrais * 0.30 * 300 - p.sum() * 15   ## 30 % convaincus, 15 EUR l'appel
    print(f"seuil {seuil} : {p.sum():>4} appels, {vrais:>3} vrais partants "
          f"-> {gain:>8.0f} euros")

**20 370 € au seuil par défaut, 28 365 € au seuil 0,20.**

Huit mille euros de plus, sans toucher une ligne du modèle. Le réglage qui
rapportait le plus n'était pas dans l'algorithme, il était dans la décision.

> ⚠️ **Ne laissez jamais `predict` choisir à votre place.** Son seuil de 0,50
> est une convention informatique, pas un arbitrage économique. Dès qu'une
> erreur coûte plus cher que l'autre, le bon seuil se calcule.

## 6. Deux erreurs, dont une qui ne prévient pas

### L'erreur bruyante

In [ ]:
LogisticRegression().fit(tel.drop(columns=["churn"]), y)   ## sans dummies

Dernière ligne :

```
ValueError: could not convert string to float: 'mensuel'
```

On a donné les colonnes de texte telles quelles. Il manque le `get_dummies`.

### L'erreur silencieuse

In [ ]:
nul = pd.Series(0, index=y_te.index)   ## "personne ne part", jamais

print("justesse :", round(100 * accuracy_score(y_te, nul), 1), "%")
print("rappel   :", round(recall_score(y_te, nul), 3))
print("F1       :", round(f1_score(y_te, nul), 3))

**73,4 % de justesse, un rappel de 0, un F1 de 0.**

Un modèle qui ne prédit jamais aucun départ affiche un chiffre parfaitement
présentable. Rien dans ce 73,4 % ne dit qu'il est inutile — et c'est
exactement pour ça qu'on ne le regarde jamais seul.

> ⚠️ **Devant tout modèle de classification, exigez quatre choses :** le score
> du modèle nul, la matrice de confusion, le rappel et le F1. Une justesse
> seule ne veut rien dire.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| du texte en colonnes numériques | `pd.get_dummies(X, drop_first=True)` |
| enchaîner mise à l'échelle et modèle | `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))` |
| une décision (0 ou 1) | `m.predict(X_te)` |
| une **probabilité** | `m.predict_proba(X_te)[:, 1]` |
| la matrice de confusion | `confusion_matrix(y_te, pred)` |
| la part de vrais parmi les prédits partants | `precision_score(y_te, pred)` |
| la part de partants retrouvés | `recall_score(y_te, pred)` |
| les deux à la fois, en un seul nombre | `f1_score(y_te, pred)` |

## La matrice de confusion, en clair

|  | prédit : reste | prédit : part |
|---|---|---|
| **reste vraiment** | vrai négatif — bien vu | **faux positif** : un appel pour rien |
| **part vraiment** | **faux négatif** : client perdu sans rien tenter | vrai positif — bien vu |

## Les quatre mesures

| Mesure | Ce qu'elle divise | Ce qu'elle répond |
|---|---|---|
| justesse | (VN + VP) / total | quelle part de mes prédictions sont bonnes ? |
| précision | VP / (VP + FP) | parmi ceux que j'appelle, combien partaient vraiment ? |
| rappel | VP / (VP + FN) | parmi ceux qui partent, combien ai-je retrouvés ? |
| F1 | 2 × précision × rappel / (précision + rappel) | les deux se tiennent-elles ensemble ? |

## Les trois phrases à retenir

1. **La justesse est un piège.** Prédire « personne ne part » donne 73,4 % de
   bonnes réponses, zéro client sauvé — et un F1 de 0.

2. **Précision et rappel arbitrent deux coûts différents.** Le rappel dit
   combien de partants on retrouve, la précision combien de contacts sont
   utiles. On ne maximise pas les deux ; le F1 dit où on en est des deux.

3. **Le seuil est une décision de gestion.** Le déplacer de 0,50 à 0,20 fait
   passer le gain de la campagne de 20 370 € à 28 365 € — sans changer une
   ligne du modèle.